# 🌍 SDG Nexus — Sustainable Development Intelligence Platform
**Built by the Diamond Blender Team**

---
### Executive Summary
Tracking the United Nations Sustainable Development Goals is hard because the data sits in
seventeen different buckets and almost never talks to itself. Our team at **Diamond Blender**
put together this notebook to fix that in one place: we pull together twelve sustainability
indicators across 150 simulated regions, line them up against the relevant SDGs, and run a
handful of machine learning models on top so a policy team can actually see what is going on
without spending a week wrangling spreadsheets.

What you get out of this notebook:
- a clean, reproducible dataset that mirrors the shape of real SDG reporting
- a risk classifier, anomaly detector and a short-term forecast for every region
- an interactive advisor and a small policy engine that turn the numbers into plain-English
  recommendations
- a final scorecard with rankings and an impact report you can drop straight into a deck

### Next Initiative — Diamond Blender Roadmap
The next thing we are working on is wiring this notebook up to live World Bank and UN data
feeds, exposing the advisor as a small web service, and shipping a lightweight mobile view
for field officers. After that we want to add a country-level deep-dive module and a public
benchmarking leaderboard so cities can compare progress year over year.


In [15]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.ensemble import RandomForestClassifier, IsolationForest, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown, clear_output
import warnings, random

warnings.filterwarnings('ignore')
np.random.seed(42); random.seed(42)
print('All libraries loaded successfully')


All libraries loaded successfully


In [16]:
# Banner — just a bit of branding for the Diamond Blender team at the top of the notebook.
BANNER = '''
<div style="background:linear-gradient(135deg,#0f2027,#203a43,#2c5364);
padding:30px;border-radius:20px;color:white;text-align:center;
box-shadow:0 10px 40px rgba(0,255,200,0.25);font-family:Segoe UI,sans-serif;">
  <h1 style="margin:0;font-size:42px;letter-spacing:2px;
  background:linear-gradient(90deg,#00f5a0,#00d9f5);-webkit-background-clip:text;
  -webkit-text-fill-color:transparent;">🌐 SDG NEXUS</h1>
  <h3 style="margin:8px 0;color:#a8f0e6;">Sustainable Development Intelligence Platform</h3>
  <p style="opacity:0.85;">Built by the Diamond Blender Team · Climate · Smart Cities · Policy · Equity</p>
</div>
'''
display(HTML(BANNER))


In [17]:
# STEP 1 — Build the practice dataset.
# Twelve indicators we care about, 150 regions, ten years of history.
# We add a small upward or downward trend per region plus some noise so the
# numbers feel like something you would actually pull from a survey.
REGIONS = [f'Region_{i+1}' for i in range(150)]
YEARS = list(range(2015, 2025))
INDICATORS = ['Poverty','Food_Access','Health_Index','Education_Index','Gender_Equality',
              'Water_Quality','Energy_Efficiency','Employment','Carbon_Emissions',
              'Smart_City_Score','Waste_Management','Renewable_Energy']

rows = []
for r in REGIONS:
    base = np.random.uniform(40, 80, len(INDICATORS))
    trend = np.random.uniform(-1.5, 2.5, len(INDICATORS))
    for y in YEARS:
        noise = np.random.normal(0, 3, len(INDICATORS))
        vals = np.clip(base + trend*(y-2015) + noise, 0, 100)
        # Note: for carbon emissions, lower is better — we flip it later when scoring.
        row = {'Region': r, 'Year': y}
        row.update(dict(zip(INDICATORS, vals)))
        rows.append(row)

df = pd.DataFrame(rows)
print(f'Generated dataset: {df.shape[0]} rows x {df.shape[1]} cols')
df.head()


Generated dataset: 1500 rows x 14 cols


,Region,Year,Poverty,Food_Access,Health_Index,Education_Index,Gender_Equality,Water_Quality,Energy_Efficiency,Employment,Carbon_Emissions,Smart_City_Score,Waste_Management,Renewable_Energy
0,Region_1,2015,59.378551,77.351243,69.482342,59.672095,44.607597,46.572549,38.870364,75.774140,62.242684,67.447822,39.018260,84.353229
1,Region_1,2016,56.770884,74.204796,70.974692,59.517426,46.584305,40.959796,38.566566,74.902546,67.207412,67.894983,40.145014,77.858530
2,Region_1,2017,54.205580,74.567753,66.352441,65.584942,46.705538,42.148712,43.751157,72.821632,63.908658,68.273883,43.253536,81.521129
3,Region_1,2018,57.953264,75.149004,67.955448,64.573829,43.954130,47.479881,39.687680,70.053176,69.324413,69.565550,39.613085,81.703335
4,Region_1,2019,63.385595,73.490639,67.273144,65.494921,45.001143,53.329815,35.375229,75.772420,68.095388,63.657783,39.772976,72.695477


In [18]:
# STEP 2 — Map each indicator to the UN SDG it actually belongs to.
# Keeping this in one dictionary so the rest of the notebook can look up the SDG label.
SDG_MAP = {
    'Poverty':'SDG 1 — No Poverty',
    'Food_Access':'SDG 2 — Zero Hunger',
    'Health_Index':'SDG 3 — Good Health',
    'Education_Index':'SDG 4 — Quality Education',
    'Gender_Equality':'SDG 5 — Gender Equality',
    'Water_Quality':'SDG 6 — Clean Water',
    'Energy_Efficiency':'SDG 7 — Affordable Energy',
    'Employment':'SDG 8 — Decent Work',
    'Carbon_Emissions':'SDG 13 — Climate Action',
    'Smart_City_Score':'SDG 11 — Sustainable Cities',
    'Waste_Management':'SDG 12 — Responsible Consumption',
    'Renewable_Energy':'SDG 7/13 — Energy & Climate'
}
sdg_df = pd.DataFrame(list(SDG_MAP.items()), columns=['Indicator','UN_SDG'])
display(sdg_df)


,Indicator,UN_SDG
0,Poverty,SDG 1 — No Poverty
1,Food_Access,SDG 2 — Zero Hunger
2,Health_Index,SDG 3 — Good Health
3,Education_Index,SDG 4 — Quality Education
4,Gender_Equality,SDG 5 — Gender Equality
5,Water_Quality,SDG 6 — Clean Water
6,Energy_Efficiency,SDG 7 — Affordable Energy
7,Employment,SDG 8 — Decent Work
8,Carbon_Emissions,SDG 13 — Climate Action
9,Smart_City_Score,SDG 11 — Sustainable Cities


In [19]:
# STEP 3 — Roll the raw indicators up into the composite scores we care about.
latest = df[df.Year == df.Year.max()].copy()

# Flip carbon so that "higher number = better" like the rest of the columns.
latest['Carbon_Score'] = 100 - latest['Carbon_Emissions']

latest['Sustainability_Score'] = latest[['Health_Index','Education_Index','Water_Quality',
    'Energy_Efficiency','Employment','Smart_City_Score','Waste_Management',
    'Renewable_Energy','Carbon_Score']].mean(axis=1)

latest['SDG_Risk_Score'] = 100 - latest['Sustainability_Score']
latest['Carbon_Reduction_Potential'] = latest['Carbon_Emissions'] * (latest['Renewable_Energy']/100)
latest['Green_Development_Index'] = (latest['Renewable_Energy'] + latest['Energy_Efficiency'] + latest['Carbon_Score'])/3
latest['Social_Equality_Index'] = (latest['Gender_Equality'] + latest['Education_Index'] + latest['Health_Index'] + (100-latest['Poverty']))/4

# Bucket each region into Low / Medium / High risk for the classifier downstream.
latest['Risk_Class'] = pd.cut(latest['SDG_Risk_Score'], bins=[-1,30,55,101],
                              labels=['Low','Medium','High'],
                              include_lowest=True, ordered=True)

# Force all three categories to exist so Plotly does not throw a KeyError later.
latest['Risk_Class'] = latest['Risk_Class'].cat.set_categories(['Low','Medium','High'], ordered=True)
print('Computed composite metrics')
latest[['Region','Sustainability_Score','SDG_Risk_Score','Green_Development_Index','Social_Equality_Index','Risk_Class']].head()


Computed composite metrics


,Region,Sustainability_Score,SDG_Risk_Score,Green_Development_Index,Social_Equality_Index,Risk_Class
9,Region_1,54.400773,45.599227,48.728792,45.921530,Medium
19,Region_2,64.641631,35.358369,70.342714,38.741355,Medium
29,Region_3,56.396432,43.603568,44.626019,48.609583,Medium
39,Region_4,61.874764,38.125236,49.852549,61.630732,Medium
49,Region_5,62.537207,37.462793,60.301534,52.837221,Medium


In [20]:
# STEP 4 — Models: a risk classifier, an outlier detector and a simple forecaster.
features = INDICATORS
X = latest[features]
y = latest['Risk_Class']

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(Xtr, ytr)
pred = clf.predict(Xte)
print(f'Risk classifier accuracy: {accuracy_score(yte,pred)*100:.2f}%')
print(classification_report(yte, pred))

# Outlier detection — flags regions that look very different from the rest of the pack.
iso = IsolationForest(contamination=0.08, random_state=42)
latest['Anomaly'] = iso.fit_predict(X)
latest['Anomaly_Flag'] = latest['Anomaly'].map({1:'Normal',-1:'Anomaly'})
print(f"Outlier regions detected: {(latest.Anomaly==-1).sum()}")

# Quick year-over-year linear forecast on the sustainability score for any single region.
def forecast_region(region, horizon=5):
    sub = df[df.Region==region].copy()
    sub['Carbon_Score'] = 100 - sub['Carbon_Emissions']
    sub['Sustainability_Score'] = sub[['Health_Index','Education_Index','Water_Quality',
        'Energy_Efficiency','Employment','Smart_City_Score','Waste_Management',
        'Renewable_Energy','Carbon_Score']].mean(axis=1)
    lr = LinearRegression().fit(sub[['Year']], sub['Sustainability_Score'])
    fut = np.arange(sub.Year.max()+1, sub.Year.max()+1+horizon).reshape(-1,1)
    return fut.flatten(), lr.predict(fut)

yrs, vals = forecast_region(REGIONS[0])
print(f'Forecast {REGIONS[0]}: ' + ', '.join(f"{y}:{v:.1f}" for y,v in zip(yrs,vals)))


Risk classifier accuracy: 94.74%
              precision    recall  f1-score   support

         Low       0.00      0.00      0.00         2
      Medium       0.95      1.00      0.97        36

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38

Outlier regions detected: 12
Forecast Region_1: 2025:54.7, 2026:54.4, 2027:54.1, 2028:53.8, 2029:53.5


In [21]:
# STEP 5 — Interactive dashboard charts.

# Leaderboard of the 20 best performing regions this year.
top = latest.nlargest(20,'Sustainability_Score')
fig = px.bar(top, x='Region', y='Sustainability_Score', color='Sustainability_Score',
             color_continuous_scale='Tealgrn', title='Top 20 Most Sustainable Regions',
             template='plotly_dark')
fig.update_layout(height=450, title_font_size=20)
fig.show()

# Correlation heatmap — useful for spotting which indicators move together.
corr = latest[features+['Sustainability_Score','SDG_Risk_Score']].corr()
fig2 = px.imshow(corr, color_continuous_scale='RdBu_r', aspect='auto',
                 title='SDG Indicator Correlation Matrix', template='plotly_dark')
fig2.update_layout(height=600); fig2.show()

# Sustainability vs risk scatter — bubble size shows green development.
# We only colour categories that actually appear in the data to avoid Plotly errors.
risk_order = [c for c in ['Low', 'Medium', 'High'] if (latest['Risk_Class'] == c).any()]
color_map = {'Low':'#00f5a0','Medium':'#00d9f5','High':'#ff4d6d'}

fig3 = px.scatter(
    latest,
    x='Sustainability_Score',
    y='SDG_Risk_Score',
    color='Risk_Class',
    size='Green_Development_Index',
    hover_data=['Region'],
    template='plotly_dark',
    title='Sustainability vs SDG Risk',
    category_orders={'Risk_Class': risk_order} if len(risk_order) else None,
    color_discrete_map={k: v for k, v in color_map.items() if k in risk_order}
)
fig3.show()


In [22]:
# STEP 6 — Radar chart per region so you can compare the shape of a region at a glance.
def radar_for(region):
    r = latest[latest.Region==region].iloc[0]
    vals = [r[i] for i in INDICATORS]
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=vals+[vals[0]], theta=INDICATORS+[INDICATORS[0]],
                                  fill='toself', line=dict(color='#00f5a0'),
                                  name=region))
    fig.update_layout(template='plotly_dark', title=f'SDG Radar — {region}',
                      polar=dict(radialaxis=dict(range=[0,100])), height=500)
    fig.show()

# Show the best and worst performers so the contrast is obvious.
radar_for(latest.nlargest(1,'Sustainability_Score').Region.iloc[0])
radar_for(latest.nsmallest(1,'Sustainability_Score').Region.iloc[0])


In [23]:
# STEP 7 — Six year forecast for the current top performers.
fig = go.Figure()
for region in latest.nlargest(5,'Sustainability_Score').Region:
    yrs, vals = forecast_region(region, horizon=6)
    fig.add_trace(go.Scatter(x=list(yrs), y=list(vals), mode='lines+markers', name=region))
fig.update_layout(template='plotly_dark', title='6-Year Sustainability Forecast (Top Regions)',
                  xaxis_title='Year', yaxis_title='Projected Sustainability Score', height=450)
fig.show()


In [24]:
# STEP 8 — Quick written summary so non-technical readers can skim the headline numbers.
def insights_summary(df_):
    best = df_.nlargest(1,'Sustainability_Score').iloc[0]
    worst = df_.nsmallest(1,'Sustainability_Score').iloc[0]
    avg = df_.Sustainability_Score.mean()
    anomalies = (df_.Anomaly==-1).sum()
    return f'''
    <div style="background:#0b1d2a;padding:22px;border-radius:14px;color:#d6f5ec;
    border-left:5px solid #00f5a0;font-family:Segoe UI;">
    <h3 style="color:#00f5a0;">Diamond Blender Insights</h3>
    <ul>
      <li><b>Top performer:</b> {best.Region} — score {best.Sustainability_Score:.1f}</li>
      <li><b>Needs attention:</b> {worst.Region} — score {worst.Sustainability_Score:.1f}</li>
      <li><b>Global average sustainability:</b> {avg:.1f}/100</li>
      <li><b>{anomalies}</b> regions flagged as outliers</li>
      <li><b>Avg Green Development Index:</b> {df_.Green_Development_Index.mean():.1f}</li>
      <li><b>Avg Social Equality Index:</b> {df_.Social_Equality_Index.mean():.1f}</li>
    </ul></div>'''
display(HTML(insights_summary(latest)))


In [25]:
# STEP 9 — Interactive advisor. Drag the sliders to describe a region and we
# return a short list of plain-English recommendations.
def recommend(profile):
    recs = []
    if profile['Poverty'] > 60: recs.append('Expand microfinance and cash-transfer programs (SDG 1).')
    if profile['Food_Access'] < 50: recs.append('Invest in climate-resilient agriculture and food banks (SDG 2).')
    if profile['Health_Index'] < 55: recs.append('Strengthen primary healthcare and telemedicine (SDG 3).')
    if profile['Education_Index'] < 55: recs.append('Digital learning plus teacher upskilling programs (SDG 4).')
    if profile['Gender_Equality'] < 60: recs.append('Enforce equal-pay policy and women leadership quotas (SDG 5).')
    if profile['Water_Quality'] < 55: recs.append('Deploy IoT water-quality sensors and treatment plants (SDG 6).')
    if profile['Renewable_Energy'] < 50: recs.append('Scale solar and wind microgrids (SDG 7).')
    if profile['Employment'] < 55: recs.append('Launch a green-jobs reskilling initiative (SDG 8).')
    if profile['Smart_City_Score'] < 55: recs.append('Implement smart traffic, lighting and waste IoT (SDG 11).')
    if profile['Waste_Management'] < 55: recs.append('Circular-economy and recycling incentives (SDG 12).')
    if profile['Carbon_Emissions'] > 60: recs.append('Carbon pricing plus EV transition roadmap (SDG 13).')
    if not recs: recs.append('Region is performing well — focus on innovation and resilience.')
    return recs

sliders = {i: widgets.IntSlider(value=60, min=0, max=100, description=i[:14]) for i in INDICATORS}
btn = widgets.Button(description='Generate Recommendations',
                     button_style='success', layout=widgets.Layout(width='320px'))
out = widgets.Output()

def on_click(b):
    profile = {k:v.value for k,v in sliders.items()}
    score = np.mean([v if k!='Carbon_Emissions' else 100-v for k,v in profile.items()])
    recs = recommend(profile)
    with out:
        clear_output()
        html = f'''<div style="background:linear-gradient(135deg,#0f2027,#2c5364);
        padding:22px;border-radius:14px;color:#eafff7;font-family:Segoe UI;">
        <h3 style="color:#00f5a0;">Sustainability Advisor</h3>
        <h4>Estimated Sustainability Score: <span style="color:#00d9f5">{score:.1f}/100</span></h4>
        <ul>{''.join(f'<li>{r}</li>' for r in recs)}</ul></div>'''
        display(HTML(html))

btn.on_click(on_click)
display(widgets.VBox(list(sliders.values()) + [btn, out]))


In [26]:
# STEP 10 — Split a hypothetical one-billion-dollar SDG budget across indicators,
# giving more money to the areas that are doing the worst.
def allocate_budget(total_budget=1_000_000_000):
    weights = (100 - latest[INDICATORS].mean(axis=0))
    weights['Carbon_Emissions'] = latest['Carbon_Emissions'].mean()  # higher emission -> more budget
    weights = weights / weights.sum()
    alloc = (weights * total_budget).round(0)
    return pd.DataFrame({'Indicator':weights.index,'Budget_USD':alloc.values,
                         'SDG':[SDG_MAP[i] for i in weights.index]}).sort_values('Budget_USD',ascending=False)

alloc_df = allocate_budget()
fig = px.treemap(alloc_df, path=['SDG','Indicator'], values='Budget_USD',
                 color='Budget_USD', color_continuous_scale='Tealgrn',
                 title='Smart Resource Allocation — $1B Global SDG Budget', template='plotly_dark')
fig.update_layout(height=550); fig.show()

# Hand-written policy ideas attached to each indicator — these are the kinds of
# moves a planning ministry would typically consider first.
POLICIES = {
 'Poverty':'Universal basic income pilots and financial inclusion mandates',
 'Food_Access':'Subsidize regenerative agriculture; national food-reserve modernization',
 'Health_Index':'Universal health coverage and modern diagnostics in rural clinics',
 'Education_Index':'Free digital learning platforms and STEM scholarships',
 'Gender_Equality':'Equal-pay enforcement and parental-leave reform',
 'Water_Quality':'Mandatory IoT water monitoring in all municipalities',
 'Energy_Efficiency':'Building retrofit mandates and appliance efficiency standards',
 'Employment':'Green-jobs guarantee program',
 'Carbon_Emissions':'Carbon tax and phase-out of fossil-fuel subsidies',
 'Smart_City_Score':'National smart-city infrastructure fund',
 'Waste_Management':'Extended producer responsibility laws',
 'Renewable_Energy':'100% renewable target by 2035 plus grid modernization'}

worst = (100-latest[INDICATORS].mean()).sort_values(ascending=False).head(5)
html = '<div style="background:#0b1d2a;padding:20px;border-radius:14px;color:#d6f5ec;font-family:Segoe UI;border-left:5px solid #00d9f5;"><h3 style="color:#00d9f5;">Policy Recommendation Engine — Top Priorities</h3><ol>'
for ind in worst.index:
    html += f'<li><b>{SDG_MAP[ind]}</b><br/><span style="opacity:.85">{POLICIES[ind]}</span></li>'
html += '</ol></div>'
display(HTML(html))


In [27]:
# STEP 11 — Tiny keyword chatbot so a non-technical user can ask a quick question
# and get a one-line pointer back. It is intentionally simple — no model required.
KB = {
 'poverty':'SDG 1 — cash transfers, microfinance, social safety nets.',
 'hunger':'SDG 2 — sustainable agriculture, food fortification, school meals.',
 'health':'SDG 3 — universal coverage, vaccination, mental-health services.',
 'education':'SDG 4 — digital classrooms, teacher training, scholarships.',
 'gender':'SDG 5 — equal pay, leadership quotas, anti-discrimination law.',
 'water':'SDG 6 — IoT monitoring, sanitation, watershed protection.',
 'energy':'SDG 7 — solar/wind, microgrids, efficiency retrofits.',
 'jobs':'SDG 8 — green-jobs program, vocational reskilling.',
 'city':'SDG 11 — smart mobility, green buildings, urban biodiversity.',
 'waste':'SDG 12 — circular economy, EPR laws, recycling incentives.',
 'climate':'SDG 13 — carbon pricing, renewables, climate adaptation.',
 'carbon':'SDG 13 — carbon tax, EV transition, reforestation.',
 'renewable':'SDG 7/13 — scale renewables, storage, smart grids.'}

def chat(q):
    q = q.lower()
    hits = [v for k,v in KB.items() if k in q]
    if not hits:
        return 'Try asking about poverty, health, education, gender, water, energy, climate, waste, cities, jobs or renewables.'
    return ' | '.join(hits)

inp = widgets.Text(placeholder='Ask the SDG Nexus assistant...', layout=widgets.Layout(width='600px'))
out2 = widgets.Output()
def ask(_):
    with out2:
        clear_output()
        display(HTML(f'<div style="background:#0b1d2a;color:#eafff7;padding:14px;border-radius:10px;">{chat(inp.value)}</div>'))
inp.on_submit(ask)
display(widgets.VBox([widgets.HTML('<h4 style="color:#00f5a0;">SDG Nexus Assistant</h4>'), inp, out2]))


In [28]:
# STEP 12 — Final scorecard, ranking and impact report.
ranking = latest.sort_values('Sustainability_Score', ascending=False).reset_index(drop=True)
ranking['Rank'] = ranking.index + 1
scorecard = ranking[['Rank','Region','Sustainability_Score','SDG_Risk_Score',
                     'Green_Development_Index','Social_Equality_Index','Risk_Class','Anomaly_Flag']].head(15)

display(HTML('<h2 style="color:#00f5a0;font-family:Segoe UI;">Final SDG Scorecard — Top 15 Regions</h2>'))
display(scorecard.style.background_gradient(cmap='Greens', subset=['Sustainability_Score','Green_Development_Index','Social_Equality_Index'])
                       .background_gradient(cmap='Reds', subset=['SDG_Risk_Score']))

report = f'''
<div style="background:linear-gradient(135deg,#0f2027,#203a43,#2c5364);
padding:30px;border-radius:18px;color:#eafff7;font-family:Segoe UI;
box-shadow:0 10px 40px rgba(0,255,200,0.2);">
  <h1 style="background:linear-gradient(90deg,#00f5a0,#00d9f5);-webkit-background-clip:text;
  -webkit-text-fill-color:transparent;">Final Impact Report</h1>
  <p><b>Regions analyzed:</b> {len(latest)} &nbsp; | &nbsp; <b>Indicators:</b> {len(INDICATORS)} &nbsp; | &nbsp; <b>SDGs covered:</b> 12+</p>
  <p><b>Global Sustainability Avg:</b> {latest.Sustainability_Score.mean():.2f}/100</p>
  <p><b>Outlier regions:</b> {(latest.Anomaly==-1).sum()}</p>
  <p><b>Green Development Index Avg:</b> {latest.Green_Development_Index.mean():.2f}</p>
  <p><b>Social Equality Index Avg:</b> {latest.Social_Equality_Index.mean():.2f}</p>
  <p><b>#1 Region:</b> {ranking.iloc[0].Region} ({ranking.iloc[0].Sustainability_Score:.2f})</p>
  <p><b>Lowest:</b> {ranking.iloc[-1].Region} ({ranking.iloc[-1].Sustainability_Score:.2f})</p>
  <hr style="border-color:#00f5a050"/>
  <h3 style="color:#00f5a0;">Closing Note from the Diamond Blender Team</h3>
  <p>This notebook brings together twelve sustainability indicators across {len(latest)} regions and
  lines them up against the UN SDGs. The classifier, outlier detector and forecasting models
  surface where attention is needed, and the advisor, policy engine, budget allocator and
  assistant translate those numbers into something a policy team can actually act on. We will
  keep iterating on it — next up is live data feeds, a public web dashboard and a country-level
  deep-dive view. Thanks for reading.</p>
</div>
'''
display(HTML(report))


,Rank,Region,Sustainability_Score,SDG_Risk_Score,Green_Development_Index,Social_Equality_Index,Risk_Class,Anomaly_Flag
0,1,Region_122,73.239080,26.760920,70.001404,58.316421,Low,Normal
1,2,Region_23,72.195916,27.804084,70.265197,61.862224,Low,Normal
2,3,Region_9,72.114700,27.885300,72.656411,59.598307,Low,Normal
3,4,Region_111,72.007339,27.992661,58.558174,76.124707,Low,Anomaly
4,5,Region_108,71.431120,28.568880,55.938399,58.902038,Low,Normal
5,6,Region_28,70.767189,29.232811,70.515077,70.122855,Low,Normal
6,7,Region_107,69.896841,30.103159,64.171878,62.703979,Medium,Normal
7,8,Region_54,69.421105,30.578895,57.476378,63.973925,Medium,Normal
8,9,Region_64,69.222811,30.777189,64.974394,56.007447,Medium,Normal
9,10,Region_116,69.112049,30.887951,60.518958,68.164010,Medium,Normal
